# Project Three, Part Two: 

## Intitial stats and visualizing the results

In [3]:
import pandas as pd
import altair as alt

df = pd.read_csv("../raw/raw_title_list - cozy_fantasy_pub_cats.csv")
df["pub_year"] = pd.to_numeric(df["pub_year"], errors="coerce")
df.info()
df["pub_category"].value_counts(normalize=True)

<class 'pandas.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           99 non-null     str    
 1   author          99 non-null     str    
 2   publisher_raw   99 non-null     str    
 3   published_date  91 non-null     str    
 4   page_count      72 non-null     float64
 5   description     81 non-null     str    
 6   cover_url       84 non-null     str    
 7   found_on_list   65 non-null     str    
 8   source          17 non-null     str    
 9   pub_year        92 non-null     float64
 10  pub_category    99 non-null     str    
dtypes: float64(2), str(9)
memory usage: 8.6 KB


pub_category
traditional       0.646465
self-published    0.353535
Name: proportion, dtype: float64

In [4]:
#looking at publication volume by year
by_year = df.groupby("pub_year").size().reset_index(name="count")

alt.Chart(by_year).mark_bar().encode(
    x="pub_year:O",
    y="count:Q"
).properties(title="Cozy fantasy titles published by year", width=500)


alt.Chart(...)

In [5]:
#Self-pub vs. traditional over time

by_year_category = df.groupby(["pub_year", "pub_category"]).size().reset_index(name="count")

alt.Chart(by_year_category).mark_area().encode(
    x="pub_year:O",
    y="count:Q",
    color="pub_category:N"
).properties(title="Self-published vs. traditional over time", width=500)


alt.Chart(...)

In [13]:
by_year_category = df.groupby(["pub_year", "pub_category"]).size().reset_index(name="count")

# re-doing chart to shade 2026 as a partial year
by_year_category["is_partial"] = by_year_category["pub_year"] == 2026

base = alt.Chart(by_year_category).mark_area().encode(
    x="pub_year:O",
    y="count:Q",
    color="pub_category:N",
    opacity=alt.condition("datum.is_partial", alt.value(0.4), alt.value(1.0))
).properties(title="Self-published vs. traditional over time", width=600)

base


alt.Chart(...)

## Reviewing Google Trends and overall genre theories

In [6]:
#Testing Google Trends for second 2024/2025 spike

from pytrends.request import TrendReq

pytrends = TrendReq()
pytrends.build_payload(
    ["cozy fantasy"],
    timeframe="2018-01-01 2026-07-01"
)
trends_df = pytrends.interest_over_time()
trends_df.reset_index(inplace=True)
trends_df.to_csv("../raw/raw_title_list - cozy_fantasy_pub_cats.csv", index=False)

alt.Chart(trends_df).mark_line().encode(
    x="date:T",
    y="cozy fantasy:Q"
).properties(title="Search interest: 'cozy fantasy'", width=500)



alt.Chart(...)

In [30]:
# cross-genre 'cozy' check

pytrends.build_payload(
    ["cozy fantasy", "cozy mystery", "cozy sci-fi", "cozy horror", "cozy games"],
    timeframe="2018-01-01 2026-07-01"
)
cross_genre_df = pytrends.interest_over_time()
cross_genre_df.reset_index(inplace=True)

cross_genre_long = cross_genre_df.melt(
    id_vars="date",
    value_vars=["cozy fantasy", "cozy mystery", "cozy sci-fi", "cozy horror", "cozy games"],
    var_name="genre", value_name="interest"
)

alt.Chart(cross_genre_long).mark_line().encode(
    x="date:T", y="interest:Q", color="genre:N"
).properties(title="The 'cozy' label across genres", width=500)


alt.Chart(...)

## Diving deeper into additional data questions

In [9]:
# how many lists mention each book? 
list_counts = df["found_on_list"].value_counts()
list_counts


found_on_list
Goodreads    38
Reddit       18
Youtube       4
Amazon        3
Penguin       2
Name: count, dtype: int64

In [31]:
from collections import Counter

# finding common 'cozy' motifs in the book descriptions
motif_words = [
    "tea", "coffee", "bakery", "cottage", "cozy", "warm", "hearth", "inn", "tavern",
    "garden", "potion", "familiar", "found family", "slow burn", "low stakes",
    "witch", "dragon", "magic school", "knitting", "baking", "herbal", "village",
    "friendship", "comfort", "gentle", "quiet", "small town", "LGBTQ", "cat", "house",
    "magic", "pastry", "whimsical", "fresh", "start", "family", "romance", "books", 
    "reading", "nature", "plants", "cat", "flower"
]

def find_motifs(description, motif_list):
    if pd.isna(description):
        return []
    text = description.lower()
    return [word for word in motif_list if word in text]

df["motifs_found"] = df["description"].apply(lambda d: find_motifs(d, motif_words))

all_motifs = [m for sublist in df["motifs_found"] for m in sublist]
Counter(all_motifs).most_common(20)

[('magic', 47),
 ('cat', 28),
 ('tea', 24),
 ('romance', 24),
 ('cozy', 21),
 ('family', 19),
 ('inn', 19),
 ('witch', 18),
 ('house', 17),
 ('start', 16),
 ('books', 15),
 ('warm', 14),
 ('dragon', 14),
 ('friendship', 9),
 ('whimsical', 8),
 ('comfort', 7),
 ('village', 7),
 ('coffee', 6),
 ('fresh', 6),
 ('quiet', 6)]

## Preparing charts for svelte integration

In [35]:
import json

# Self-pub vs. traditional, by year
pivot = df.groupby(["pub_year", "pub_category"]).size().unstack(fill_value=0).reset_index()
pivot = pivot.rename(columns={"pub_year": "year"})
pivot.to_json("../raw/self_pub_by_year.json", orient="records")

# Total publication count, by year, for the timeline
timeline = df.groupby("pub_year").size().reset_index(name="count")
timeline = timeline.rename(columns={"pub_year": "year"})
timeline.to_json("../raw/timeline_counts.json", orient="records")


In [36]:
pivot.columns

Index(['year', 'self-published', 'traditional'], dtype='str', name='pub_category')